# 🚀 GPT-CUDA v2: Fixed Transformer from Scratch in C++/CUDA

**Self-contained notebook** — writes all source files, compiles, trains, generates text.

### Fixes over v1:
1. 🔴 **ReLU/GELU actually applied** (was commented out — FFN was linear!)
2. 🔴 **RMSNorm gamma learnable** with Adam (was frozen at 1.0)
3. 🟠 **Xavier/Glorot init** on GPU (was uniform [-0.05, 0.05])
4. 🟡 **LR warmup + cosine decay** (was constant 3e-4)
5. 🟢 **GELU instead of ReLU** in FFN (modern standard)
6. 📊 **CSV training log** for plotting

**Run on:** Colab T4, Thunder Compute, any CUDA GPU
**Expected:** loss drops from ~4.0 → ~0.5 over 10K steps

In [ ]:
!nvidia-smi

In [ ]:
!wget -q https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt -O input.txt
!wc -c input.txt
print('Data downloaded ✓')

In [ ]:
%%writefile Layer.cuh
#pragma once
#include <cublas_v2.h>

#define ACTIVATION_NONE 0
#define ACTIVATION_RELU 1
#define ACTIVATION_GELU 2

class Layer {
  protected:
    int batch_features, in_features, out_features;
  public:
    Layer(int bs, int in_f, int out_f) : batch_features(bs), in_features(in_f), out_features(out_f) {}
    virtual ~Layer() {}
    virtual float* forward(cublasHandle_t h, void* inp, int act) = 0;
    virtual float* backward(cublasHandle_t h, float* dY) = 0;
    virtual void step(float lr, int t) = 0;
};


In [ ]:
%%writefile LinearLayer.cuh
#pragma once
#include "Layer.cuh"
class LinearLayer : public Layer {
  float *d_W, *d_b, *d_dW, *d_db, *d_X_cache, *d_Y, *d_dX, *d_m, *d_v;
public:
  LinearLayer(int bs, int in_f, int out_f);
  ~LinearLayer();
  float* forward(cublasHandle_t h, void* inp, int act) override;
  float* backward(cublasHandle_t h, float* dY) override;
  void step(float lr, int t) override;
};


In [ ]:
%%writefile LinearLayer.cu
#include "LinearLayer.cuh"
#include <cmath>
#include <cstdlib>

// ─── Xavier/Glorot uniform initialization ──────────────────────────────
__global__ void xavier_init_kernel(float* W, int in_f, int out_f, unsigned int seed) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    int total = in_f * out_f;
    if (idx < total) {
        float limit = sqrtf(6.0f / (float)(in_f + out_f));
        // Simple pseudo-random on GPU using thread index + seed
        unsigned int s = idx + seed;
        s = (s ^ 61) ^ (s >> 16);
        s = s * 9;
        s = s ^ (s >> 4);
        s = s * 0x27d4eb2d;
        s = s ^ (s >> 15);
        float r = (float)(s & 0xFFFF) / 65535.0f;
        W[idx] = (2.0f * r - 1.0f) * limit;
    }
}

// ─── FIX #1: ReLU activation kernel ────────────────────────────────────
__global__ void relu_forward_kernel(float* d_Y, int total) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < total) {
        if (d_Y[idx] < 0.0f) d_Y[idx] = 0.0f;
    }
}

__global__ void relu_backward_kernel(float* d_dY, float* d_Y, int total) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < total) {
        if (d_Y[idx] <= 0.0f) d_dY[idx] = 0.0f;
    }
}

// ─── FIX #5: GELU activation kernel (approximation) ────────────────────
__global__ void gelu_forward_kernel(float* d_Y, int total) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < total) {
        float x = d_Y[idx];
        float cdf = 0.5f * (1.0f + tanhf(0.7978845608f * (x + 0.044715f * x * x * x)));
        d_Y[idx] = x * cdf;
    }
}

__global__ void gelu_backward_kernel(float* d_dY, float* d_Y_save, int total) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < total) {
        float x = d_Y_save[idx];
        float x3 = x * x * x;
        float tanh_arg = 0.7978845608f * (x + 0.044715f * x3);
        float tanh_val = tanhf(tanh_arg);
        float sech2 = 1.0f - tanh_val * tanh_val;
        float gelu_deriv = 0.5f * (1.0f + tanh_val) + 0.5f * x * 0.7978845608f * (1.0f + 3.0f * 0.044715f * x * x) * sech2;
        d_dY[idx] *= gelu_deriv;
    }
}

// ─── AdamW kernel ───────────────────────────────────────────────────────
__global__ void linear_adam_kernel(float* W, float* dW, float* m, float* v, float lr, int t, int size) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < size) {
        float b1 = 0.9f, b2 = 0.999f, eps = 1e-8f, wd = 0.01f;
        m[idx] = b1 * m[idx] + (1.0f - b1) * dW[idx];
        v[idx] = b2 * v[idx] + (1.0f - b2) * (dW[idx] * dW[idx]);
        float m_hat = m[idx] / (1.0f - powf(b1, (float)t));
        float v_hat = v[idx] / (1.0f - powf(b2, (float)t));
        W[idx] -= lr * wd * W[idx];
        W[idx] -= lr * m_hat / (sqrtf(v_hat) + eps);
        dW[idx] = 0.0f;
    }
}

__global__ void add_bias_kernel(float* Y, const float* b, int batch_f, int out_f) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < batch_f * out_f) {
        Y[idx] += b[idx % out_f];
    }
}

// ─── Constructor with Xavier init ───────────────────────────────────────
LinearLayer::LinearLayer(int bs, int in_f, int out_f) : Layer(bs, in_f, out_f) {
    cudaMalloc(&d_W, sizeof(float) * in_f * out_f);
    cudaMalloc(&d_b, sizeof(float) * out_f);
    cudaMalloc(&d_dW, sizeof(float) * in_f * out_f);
    cudaMalloc(&d_db, sizeof(float) * out_f);
    cudaMalloc(&d_Y, sizeof(float) * bs * out_f);
    cudaMalloc(&d_dX, sizeof(float) * bs * in_f);

    // FIX #3: Xavier/Glorot uniform initialization
    int threads = 256;
    int blocks = (in_f * out_f + threads - 1) / threads;
    xavier_init_kernel<<<blocks, threads>>>(d_W, in_f, out_f, (unsigned int)time(0) + in_f * 31 + out_f * 17);
    cudaMemset(d_b, 0, sizeof(float) * out_f);

    cudaMalloc(&d_m, sizeof(float) * in_f * out_f);
    cudaMalloc(&d_v, sizeof(float) * in_f * out_f);
    cudaMemset(d_m, 0, sizeof(float) * in_f * out_f);
    cudaMemset(d_v, 0, sizeof(float) * in_f * out_f);
}

LinearLayer::~LinearLayer() {
    cudaFree(d_W); cudaFree(d_b); cudaFree(d_dW); cudaFree(d_db);
    cudaFree(d_Y); cudaFree(d_dX); cudaFree(d_m); cudaFree(d_v);
}

float* LinearLayer::forward(cublasHandle_t h, void* inp, int act) {
    d_X_cache = (float*)inp;
    const float alpha = 1.0f, beta = 0.0f;

    cublasSgemm(h, CUBLAS_OP_N, CUBLAS_OP_N,
                out_features, batch_features, in_features,
                &alpha, d_W, out_features, d_X_cache, in_features,
                &beta, d_Y, out_features);

    int threads = 256;
    int blocks_bias = (batch_features * out_features + threads - 1) / threads;
    add_bias_kernel<<<blocks_bias, threads>>>(d_Y, d_b, batch_features, out_features);

    // FIX #1: Actually APPLY the activation!
    if (act == ACTIVATION_RELU) {
        int total = batch_features * out_features;
        int blocks_act = (total + threads - 1) / threads;
        relu_forward_kernel<<<blocks_act, threads>>>(d_Y, total);
    } else if (act == ACTIVATION_GELU) {
        int total = batch_features * out_features;
        int blocks_act = (total + threads - 1) / threads;
        gelu_forward_kernel<<<blocks_act, threads>>>(d_Y, total);
    }

    return d_Y;
}

float* LinearLayer::backward(cublasHandle_t h, float* d_dY) {
    const float alpha = 1.0f, beta = 0.0f;

    // dX = dY * W^T
    cublasSgemm(h, CUBLAS_OP_T, CUBLAS_OP_N,
                in_features, batch_features, out_features,
                &alpha, d_W, out_features, d_dY, out_features,
                &beta, d_dX, in_features);

    // dW = X^T * dY
    cublasSgemm(h, CUBLAS_OP_N, CUBLAS_OP_T,
                out_features, in_features, batch_features,
                &alpha, d_dY, out_features, d_X_cache, in_features,
                &beta, d_dW, out_features);

    return d_dX;
}

void LinearLayer::step(float lr, int t) {
    int total = in_features * out_features;
    int threads = 256;
    int blocks = (total + threads - 1) / threads;
    linear_adam_kernel<<<blocks, threads>>>(d_W, d_dW, d_m, d_v, lr, t, total);
}


In [ ]:
%%writefile EmbeddingLayer.cuh
#pragma once
#include "Layer.cuh"

class EmbeddingLayer : public Layer {
  private:
    // Poids et gradients
    float* d_W;  // Le dictionnaire [vocab_size * embedding_dim]
    float* d_dW; // Le gradient des poids

    // Encodage Positionnel (Constant)
    float* d_PE; // La matrice des ondes [context_size * embedding_dim]
    
    // Entrées / Sorties
    int* d_X;    // L'entrée (Tableau d'entiers) [batch_size * context_size]
    float* d_Y;  // La sortie [batch_size * context_size * embedding_dim]
    
    // Les dimensions
    int vocab_size;
    int embedding_dim;
    int batch_size;
    int context_size;

    //ADAM
    float* d_m;
    float* d_v;
    
  public:
    EmbeddingLayer(int vocab_size, int embedding_dim, int batch_size, int context_size);
    ~EmbeddingLayer();

    float* forward(cublasHandle_t handle, void* d_input, int activation_type) override; 
    float* backward(cublasHandle_t handle, float* d_dY) override;
    void step(float learning_rate,int t) override;
};


In [ ]:
%%writefile EmbeddingLayer.cu
#include "EmbeddingLayer.cuh"
#include <cmath>
#include <cstdlib>

// Re-use the xavier init kernel (declare extern)
__global__ void xavier_init_kernel(float* W, int in_f, int out_f, unsigned int seed);

__global__ void embedding_forward_kernel(int* X, float* W, float* PE, float* Y, int emb_dim, int cs, int total) {
    int idx = blockDim.x * blockIdx.x + threadIdx.x;
    if (idx < total) {
        int e_idx = idx % emb_dim;
        int w_pos = idx / emb_dim;
        int s_pos = w_pos % cs;
        int v_id = X[w_pos];
        Y[idx] = W[v_id * emb_dim + e_idx] + PE[s_pos * emb_dim + e_idx];
    }
}

__global__ void backward_embedding_kernel(int* X, float* dY, float* dW, int emb_dim, int total) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < total) {
        int e_idx = idx % emb_dim;
        int w_pos = idx / emb_dim;
        int v_id = X[w_pos];
        atomicAdd(&dW[v_id * emb_dim + e_idx], dY[idx]);
    }
}

__global__ void embedding_adam_kernel(float* W, float* dW, float* m, float* v, float lr, int t, int size) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < size) {
        float b1 = 0.9f, b2 = 0.999f, eps = 1e-8f, wd = 0.01f;
        m[idx] = b1 * m[idx] + (1.0f - b1) * dW[idx];
        v[idx] = b2 * v[idx] + (1.0f - b2) * (dW[idx] * dW[idx]);
        float m_hat = m[idx] / (1.0f - powf(b1, (float)t));
        float v_hat = v[idx] / (1.0f - powf(b2, (float)t));
        W[idx] -= lr * wd * W[idx];
        W[idx] -= lr * m_hat / (sqrtf(v_hat) + eps);
        dW[idx] = 0.0f;
    }
}

EmbeddingLayer::EmbeddingLayer(int vs, int ed, int bs, int cs) : Layer(bs, cs, ed) {
    vocab_size = vs; embedding_dim = ed; batch_size = bs; context_size = cs;
    cudaMalloc(&d_W, sizeof(float) * vs * ed);
    cudaMalloc(&d_dW, sizeof(float) * vs * ed);
    cudaMalloc(&d_Y, sizeof(float) * bs * cs * ed);
    cudaMalloc(&d_PE, sizeof(float) * cs * ed);

    // FIX #3: Xavier init on GPU
    int threads = 256;
    int blocks = (vs * ed + threads - 1) / threads;
    xavier_init_kernel<<<blocks, threads>>>(d_W, vs, ed, 42);

    // Positional encoding on CPU
    float* h_PE = (float*)malloc(sizeof(float) * cs * ed);
    for (int pos = 0; pos < cs; pos++) {
        for (int i = 0; i < ed; i += 2) {
            float div = pow(10000.0f, (float)i / ed);
            h_PE[pos * ed + i] = sin(pos / div);
            if (i + 1 < ed) h_PE[pos * ed + i + 1] = cos(pos / div);
        }
    }
    cudaMemcpy(d_PE, h_PE, sizeof(float) * cs * ed, cudaMemcpyHostToDevice);
    free(h_PE);

    cudaMalloc(&d_m, sizeof(float) * vs * ed);
    cudaMalloc(&d_v, sizeof(float) * vs * ed);
    cudaMemset(d_m, 0, sizeof(float) * vs * ed);
    cudaMemset(d_v, 0, sizeof(float) * vs * ed);
}

EmbeddingLayer::~EmbeddingLayer() {
    cudaFree(d_W); cudaFree(d_dW); cudaFree(d_Y); cudaFree(d_PE); cudaFree(d_m); cudaFree(d_v);
}

float* EmbeddingLayer::forward(cublasHandle_t h, void* inp, int act) {
    d_X = (int*)inp;
    int total = batch_size * context_size;
    int tpb = 256;
    int bpg = (total + tpb - 1) / tpb;
    embedding_forward_kernel<<<bpg, tpb>>>(d_X, d_W, d_PE, d_Y, embedding_dim, context_size, total);
    cudaDeviceSynchronize();
    return d_Y;
}

float* EmbeddingLayer::backward(cublasHandle_t h, float* d_dY) {
    int total = batch_size * context_size * embedding_dim;
    int tpb = 256;
    int bpg = (total + tpb - 1) / tpb;
    cudaMemset(d_dW, 0, sizeof(float) * vocab_size * embedding_dim);
    backward_embedding_kernel<<<bpg, tpb>>>(d_X, d_dY, d_dW, embedding_dim, total);
    cudaDeviceSynchronize();
    return nullptr;
}

void EmbeddingLayer::step(float lr, int t) {
    int total = vocab_size * embedding_dim;
    int threads = 256;
    int blocks = (total + threads - 1) / threads;
    embedding_adam_kernel<<<blocks, threads>>>(d_W, d_dW, d_m, d_v, lr, t, total);
}


In [ ]:
%%writefile RMSNormLayer.cuh
#pragma once
#include "Layer.cuh"

class RMSNormLayer : public Layer {
  private:
    float* d_gamma;
    
    // Les tampons pour le backward bridé
    float* d_inv_rms; // Sauvegarde de la division (l'échelle)
    float* d_dX;      // Le gradient de sortie corrigé
    
    int batch_size;
    int context_size;
    int embedding_dim;

    float* d_Y;
    float* d_X_cache;

  public:
    // Le constructeur exact que le .cu va chercher
    RMSNormLayer(int batch_size, int context_size, int embedding_dim);
    ~RMSNormLayer();

    float* forward(cublasHandle_t handle, void* d_input, int activation_type) override; 
    float* backward(cublasHandle_t handle, float* d_dY) override;
    void step(float learning_rate, int t) override;
};


In [ ]:
%%writefile RMSNormLayer.cu
#include "RMSNormLayer.cuh"
#include <cmath>

__global__ void rmsnorm_forward_kernel(float* X, float* Y, float* gamma, float* inv_rms, int emb_dim, float eps) {
    int row = blockIdx.x, tid = threadIdx.x;
    extern __shared__ float shared_sq[];
    float val = X[row * emb_dim + tid];
    shared_sq[tid] = val * val;
    __syncthreads();
    for (int s = blockDim.x/2; s > 0; s >>= 1) {
        if (tid < s) shared_sq[tid] += shared_sq[tid + s];
        __syncthreads();
    }
    if (tid == 0) {
        shared_sq[0] = rsqrtf((shared_sq[0] / emb_dim) + eps);
        inv_rms[row] = shared_sq[0];
    }
    __syncthreads();
    Y[row * emb_dim + tid] = val * shared_sq[0] * gamma[tid];
}

__global__ void rmsnorm_backward_kernel(float* dY, float* X, float* dX, float* inv_rms, float* gamma, float* dgamma, int emb_dim) {
    int row = blockIdx.x, tid = threadIdx.x;
    extern __shared__ float shared_dot[];
    float dy = dY[row * emb_dim + tid];
    float x  = X[row * emb_dim + tid];
    float g  = gamma[tid];
    shared_dot[tid] = dy * x * g;
    __syncthreads();
    for (int s = blockDim.x/2; s > 0; s >>= 1) {
        if (tid < s) shared_dot[tid] += shared_dot[tid + s];
        __syncthreads();
    }
    float dot_sum = shared_dot[0];
    float inv = inv_rms[row];
    float scale = (inv * inv * inv) / emb_dim;
    // Gradient for X
    dX[row * emb_dim + tid] = dy * inv * g - x * dot_sum * scale;
    // FIX #2: Gradient for gamma
    dgamma[tid] = dy * x * inv;
}

// Adam kernel for 1D parameter (gamma)
__global__ void gamma_adam_kernel(float* gamma, float* dgamma, float* m, float* v,
                                   float lr, int t, int size) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < size) {
        float b1 = 0.9f, b2 = 0.999f, eps = 1e-8f;
        m[idx] = b1 * m[idx] + (1.0f - b1) * dgamma[idx];
        v[idx] = b2 * v[idx] + (1.0f - b2) * (dgamma[idx] * dgamma[idx]);
        float m_hat = m[idx] / (1.0f - powf(b1, (float)t));
        float v_hat = v[idx] / (1.0f - powf(b2, (float)t));
        gamma[idx] -= lr * m_hat / (sqrtf(v_hat) + eps);
        dgamma[idx] = 0.0f;
    }
}

RMSNormLayer::RMSNormLayer(int bs, int cs, int ed) : Layer(bs, cs, ed) {
    batch_size = bs; context_size = cs; embedding_dim = ed;
    cudaMalloc(&d_gamma, sizeof(float) * ed);
    cudaMalloc(&d_dgamma, sizeof(float) * ed);
    cudaMalloc(&d_m_gamma, sizeof(float) * ed);
    cudaMalloc(&d_v_gamma, sizeof(float) * ed);
    cudaMalloc(&d_inv_rms, sizeof(float) * bs * cs);
    cudaMalloc(&d_dX, sizeof(float) * bs * cs * ed);
    cudaMalloc(&d_Y, sizeof(float) * bs * cs * ed);
    float* h_gamma = (float*)malloc(sizeof(float) * ed);
    for (int i = 0; i < ed; i++) h_gamma[i] = 1.0f;
    cudaMemcpy(d_gamma, h_gamma, sizeof(float) * ed, cudaMemcpyHostToDevice);
    cudaMemset(d_dgamma, 0, sizeof(float) * ed);
    cudaMemset(d_m_gamma, 0, sizeof(float) * ed);
    cudaMemset(d_v_gamma, 0, sizeof(float) * ed);
    free(h_gamma);
}

RMSNormLayer::~RMSNormLayer() {
    cudaFree(d_gamma); cudaFree(d_dgamma); cudaFree(d_m_gamma); cudaFree(d_v_gamma);
    cudaFree(d_inv_rms); cudaFree(d_dX); cudaFree(d_Y);
}

float* RMSNormLayer::forward(cublasHandle_t h, void* inp, int act) {
    d_X_cache = (float*)inp;
    int total = batch_size * context_size;
    rmsnorm_forward_kernel<<<total, embedding_dim, embedding_dim * sizeof(float)>>>(
        d_X_cache, d_Y, d_gamma, d_inv_rms, embedding_dim, 1e-5f);
    cudaDeviceSynchronize();
    return d_Y;
}

float* RMSNormLayer::backward(cublasHandle_t h, float* d_dY) {
    int total = batch_size * context_size;
    // Zero dgamma before accumulation
    cudaMemset(d_dgamma, 0, sizeof(float) * embedding_dim);
    rmsnorm_backward_kernel<<<total, embedding_dim, embedding_dim * sizeof(float)>>>(
        d_dY, d_X_cache, d_dX, d_inv_rms, d_gamma, d_dgamma, embedding_dim);
    cudaDeviceSynchronize();
    return d_dX;
}

void RMSNormLayer::step(float lr, int t) {
    // FIX #2: Update gamma with Adam!
    int threads = 256;
    int blocks = (embedding_dim + threads - 1) / threads;
    gamma_adam_kernel<<<blocks, threads>>>(d_gamma, d_dgamma, d_m_gamma, d_v_gamma, lr, t, embedding_dim);
}


In [ ]:
%%writefile AttentionLayer.cuh
#pragma once
#include "Layer.cuh"
#include "LinearLayer.cuh"
class AttentionLayer : public Layer {
  LinearLayer *W_q, *W_k, *W_v, *W_o;
  float *d_Q, *d_K, *d_V, *d_Scores, *d_AttentionOut;
  float *d_Q_grad, *d_K_grad, *d_V_grad, *d_dScores_softmax, *d_dScores, *d_dX;
  int batch_size, context_size, embedding_dim;
public:
  AttentionLayer(int bs, int cs, int ed);
  ~AttentionLayer();
  float* forward(cublasHandle_t h, void* inp, int act) override;
  float* backward(cublasHandle_t h, float* dY) override;
  void step(float lr, int t) override;
};


In [ ]:
%%writefile AttentionLayer.cu
#include "AttentionLayer.cuh"
#include <cmath>

__global__ void causal_mask_kernel(float* Scores, int cs, int bs) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    int total = bs * cs * cs;
    if (idx < total) {
        int m_idx = idx % (cs * cs);
        int row = m_idx / cs, col = m_idx % cs;
        if (col > row) Scores[idx] = -1e9f;
    }
}

__global__ void scale_scores_kernel(float* Scores, float scale, int total) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < total) Scores[idx] *= scale;
}

__global__ void softmax_forward_kernel(float* Scores, int cs, int bs) {
    int row = blockIdx.x;
    int tid = threadIdx.x;
    extern __shared__ float shared[];
    if (row < bs * cs && tid < cs) {
        int base = row * cs;
        float max_val = -1e9f;
        for (int i = 0; i < cs; i++) max_val = fmaxf(max_val, Scores[base + i]);
        float my_exp = expf(Scores[base + tid] - max_val);
        shared[tid] = my_exp;
        __syncthreads();
        float sum = 0.0f;
        for (int i = 0; i < cs; i++) sum += shared[i];
        Scores[base + tid] = my_exp / (sum + 1e-9f);
    }
}

__global__ void softmax_backward_kernel(float* dSoft, float* Soft, float* dScores, int cs) {
    int row = blockIdx.x, tid = threadIdx.x;
    if (tid < cs) {
        int base = row * cs;
        float sum = 0.0f;
        for (int i = 0; i < cs; i++) sum += dSoft[base + i] * Soft[base + i];
        int idx = base + tid;
        dScores[idx] = Soft[idx] * (dSoft[idx] - sum);
    }
}

__global__ void sum_gradients_kernel(float* dX, const float* dXq, const float* dXk, const float* dXv, int total) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < total) dX[idx] = dXq[idx] + dXk[idx] + dXv[idx];
}

AttentionLayer::AttentionLayer(int bs, int cs, int ed) : Layer(bs, cs, ed) {
    batch_size = bs; context_size = cs; embedding_dim = ed;
    int fb = bs * cs;
    W_q = new LinearLayer(fb, ed, ed);
    W_k = new LinearLayer(fb, ed, ed);
    W_v = new LinearLayer(fb, ed, ed);
    W_o = new LinearLayer(fb, ed, ed);
    cudaMalloc(&d_Scores, sizeof(float) * bs * cs * cs);
    cudaMalloc(&d_AttentionOut, sizeof(float) * bs * cs * ed);
    int te = bs * cs * ed, ts = bs * cs * cs;
    cudaMalloc(&d_Q_grad, sizeof(float) * te);
    cudaMalloc(&d_K_grad, sizeof(float) * te);
    cudaMalloc(&d_V_grad, sizeof(float) * te);
    cudaMalloc(&d_dX, sizeof(float) * te);
    cudaMalloc(&d_dScores_softmax, sizeof(float) * ts);
    cudaMalloc(&d_dScores, sizeof(float) * ts);
}
AttentionLayer::~AttentionLayer() {
    delete W_q; delete W_k; delete W_v; delete W_o;
    cudaFree(d_Scores); cudaFree(d_AttentionOut);
    cudaFree(d_Q_grad); cudaFree(d_K_grad); cudaFree(d_V_grad);
    cudaFree(d_dX); cudaFree(d_dScores_softmax); cudaFree(d_dScores);
}

float* AttentionLayer::forward(cublasHandle_t h, void* inp, int act) {
    float* X = (float*)inp;
    const float alpha = 1.0f, beta = 0.0f;

    d_Q = W_q->forward(h, X, ACTIVATION_NONE);
    d_K = W_k->forward(h, X, ACTIVATION_NONE);
    d_V = W_v->forward(h, X, ACTIVATION_NONE);

    long long strideE = context_size * embedding_dim;
    long long strideS = context_size * context_size;

    // Scores = Q @ K^T
    cublasSgemmStridedBatched(h, CUBLAS_OP_T, CUBLAS_OP_N,
        context_size, context_size, embedding_dim,
        &alpha, d_K, embedding_dim, strideE, d_Q, embedding_dim, strideE,
        &beta, d_Scores, context_size, strideS, batch_size);

    int totalS = batch_size * context_size * context_size;
    int t256 = 256;
    int bS = (totalS + t256 - 1) / t256;
    scale_scores_kernel<<<bS, t256>>>(d_Scores, 1.0f/sqrtf((float)embedding_dim), totalS);
    causal_mask_kernel<<<bS, t256>>>(d_Scores, context_size, batch_size);
    cudaDeviceSynchronize();

    int bSoft = batch_size * context_size;
    softmax_forward_kernel<<<bSoft, context_size, context_size*sizeof(float)>>>(d_Scores, context_size, batch_size);
    cudaDeviceSynchronize();

    // Out = Scores @ V
    cublasSgemmStridedBatched(h, CUBLAS_OP_N, CUBLAS_OP_N,
        embedding_dim, context_size, context_size,
        &alpha, d_V, embedding_dim, strideE, d_Scores, context_size, strideS,
        &beta, d_AttentionOut, embedding_dim, strideE, batch_size);

    return W_o->forward(h, d_AttentionOut, ACTIVATION_NONE);
}

float* AttentionLayer::backward(cublasHandle_t h, float* dY) {
    const float alpha = 1.0f, beta = 0.0f;
    long long strideE = context_size * embedding_dim;
    long long strideS = context_size * context_size;

    float* dOut = W_o->backward(h, dY);

    // dV, dScores_softmax
    cublasSgemmStridedBatched(h, CUBLAS_OP_N, CUBLAS_OP_T,
        embedding_dim, context_size, context_size,
        &alpha, dOut, embedding_dim, strideE, d_Scores, context_size, strideS,
        &beta, d_V_grad, embedding_dim, strideE, batch_size);

    cublasSgemmStridedBatched(h, CUBLAS_OP_T, CUBLAS_OP_N,
        context_size, context_size, embedding_dim,
        &alpha, d_V, embedding_dim, strideE, dOut, embedding_dim, strideE,
        &beta, d_dScores_softmax, context_size, strideS, batch_size);

    // Softmax backward
    int bSoft = batch_size * context_size;
    softmax_backward_kernel<<<bSoft, context_size>>>(d_dScores_softmax, d_Scores, d_dScores, context_size);
    cudaDeviceSynchronize();

    // Scale backward
    int totalS = batch_size * context_size * context_size;
    int t256 = 256;
    int bS = (totalS + t256 - 1) / t256;
    scale_scores_kernel<<<bS, t256>>>(d_dScores, 1.0f/sqrtf((float)embedding_dim), totalS);

    // dQ, dK from dScores
    cublasSgemmStridedBatched(h, CUBLAS_OP_N, CUBLAS_OP_N,
        embedding_dim, context_size, context_size,
        &alpha, d_K, embedding_dim, strideE, d_dScores, context_size, strideS,
        &beta, d_Q_grad, embedding_dim, strideE, batch_size);

    cublasSgemmStridedBatched(h, CUBLAS_OP_N, CUBLAS_OP_T,
        embedding_dim, context_size, context_size,
        &alpha, d_Q, embedding_dim, strideE, d_dScores, context_size, strideS,
        &beta, d_K_grad, embedding_dim, strideE, batch_size);

    float* dXq = W_q->backward(h, d_Q_grad);
    float* dXk = W_k->backward(h, d_K_grad);
    float* dXv = W_v->backward(h, d_V_grad);

    int totalE = batch_size * context_size * embedding_dim;
    int bE = (totalE + t256 - 1) / t256;
    sum_gradients_kernel<<<bE, t256>>>(d_dX, dXq, dXk, dXv, totalE);
    cudaDeviceSynchronize();
    return d_dX;
}

void AttentionLayer::step(float lr, int t) {
    W_q->step(lr, t); W_k->step(lr, t); W_v->step(lr, t); W_o->step(lr, t);
}


In [ ]:
%%writefile FeedForwardLayer.cuh
#pragma once
#include "Layer.cuh"
#include "LinearLayer.cuh"
class FeedForwardLayer : public Layer {
  LinearLayer *fc1, *fc2;
  int batch_size, context_size, embedding_dim, hidden_dim;
public:
  FeedForwardLayer(int bs, int cs, int ed, int exp=4);
  ~FeedForwardLayer();
  float* forward(cublasHandle_t h, void* inp, int act) override;
  float* backward(cublasHandle_t h, float* dY) override;
  void step(float lr, int t) override;
};


In [ ]:
%%writefile FeedForwardLayer.cu
#include "FeedForwardLayer.cuh"

FeedForwardLayer::FeedForwardLayer(int bs, int cs, int ed, int exp) : Layer(bs, cs, ed) {
    batch_size = bs; context_size = cs; embedding_dim = ed;
    hidden_dim = ed * exp;
    int fb = bs * cs;
    fc1 = new LinearLayer(fb, ed, hidden_dim);
    fc2 = new LinearLayer(fb, hidden_dim, ed);
}
FeedForwardLayer::~FeedForwardLayer() { delete fc1; delete fc2; }

float* FeedForwardLayer::forward(cublasHandle_t h, void* inp, int act) {
    // FIX #5: Use GELU instead of ReLU (modern transformer standard)
    float* hidden = fc1->forward(h, (float*)inp, ACTIVATION_GELU);
    return fc2->forward(h, hidden, ACTIVATION_NONE);
}
float* FeedForwardLayer::backward(cublasHandle_t h, float* dY) {
    float* dHidden = fc2->backward(h, dY);
    return fc1->backward(h, dHidden);
}
void FeedForwardLayer::step(float lr, int t) {
    fc1->step(lr, t); fc2->step(lr, t);
}


In [ ]:
%%writefile TransformerBlock.cuh
#pragma once
#include "Layer.cuh"
#include "AttentionLayer.cuh"
#include "FeedForwardLayer.cuh"
#include "RMSNormLayer.cuh"
class TransformerBlock : public Layer {
  RMSNormLayer *norm1, *norm2;
  AttentionLayer *attn;
  FeedForwardLayer *ffn;
  float *d_res1, *d_out, *d_dRes1, *d_dX;
  int batch_size, context_size, embedding_dim;
public:
  TransformerBlock(int bs, int cs, int ed);
  ~TransformerBlock();
  float* forward(cublasHandle_t h, void* inp, int act) override;
  float* backward(cublasHandle_t h, float* dY) override;
  void step(float lr, int t) override;
};


In [ ]:
%%writefile TransformerBlock.cu
#include "TransformerBlock.cuh"

__global__ void add_tensors_kernel(float* A, float* B, float* C, int size) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < size) C[idx] = A[idx] + B[idx];
}

TransformerBlock::TransformerBlock(int bs, int cs, int ed) : Layer(bs, cs, ed) {
    batch_size = bs; context_size = cs; embedding_dim = ed;
    norm1 = new RMSNormLayer(bs, cs, ed);
    attn  = new AttentionLayer(bs, cs, ed);
    norm2 = new RMSNormLayer(bs, cs, ed);
    ffn   = new FeedForwardLayer(bs, cs, ed, 4);
    int total = bs * cs * ed;
    cudaMalloc(&d_res1, sizeof(float) * total);
    cudaMalloc(&d_out,  sizeof(float) * total);
    cudaMalloc(&d_dRes1,sizeof(float) * total);
    cudaMalloc(&d_dX,   sizeof(float) * total);
}
TransformerBlock::~TransformerBlock() {
    delete norm1; delete attn; delete norm2; delete ffn;
    cudaFree(d_res1); cudaFree(d_out); cudaFree(d_dRes1); cudaFree(d_dX);
}

float* TransformerBlock::forward(cublasHandle_t h, void* inp, int act) {
    float* X = (float*)inp;
    int total = batch_size * context_size * embedding_dim;
    int t = 256;
    int b = (total + t - 1) / t;

    float* n1 = norm1->forward(h, X, ACTIVATION_NONE);
    float* a  = attn->forward(h, n1, ACTIVATION_NONE);
    add_tensors_kernel<<<b, t>>>(X, a, d_res1, total);
    cudaDeviceSynchronize();

    float* n2 = norm2->forward(h, d_res1, ACTIVATION_NONE);
    float* f  = ffn->forward(h, n2, ACTIVATION_NONE);
    add_tensors_kernel<<<b, t>>>(d_res1, f, d_out, total);
    cudaDeviceSynchronize();

    return d_out;
}

float* TransformerBlock::backward(cublasHandle_t h, float* dY) {
    int total = batch_size * context_size * embedding_dim;
    int t = 256;
    int b = (total + t - 1) / t;

    float* dF = ffn->backward(h, dY);
    float* dN2 = norm2->backward(h, dF);
    add_tensors_kernel<<<b, t>>>(dY, dN2, d_dRes1, total);
    cudaDeviceSynchronize();

    float* dA = attn->backward(h, d_dRes1);
    float* dN1 = norm1->backward(h, dA);
    add_tensors_kernel<<<b, t>>>(d_dRes1, dN1, d_dX, total);
    cudaDeviceSynchronize();

    return d_dX;
}

void TransformerBlock::step(float lr, int t) {
    norm1->step(lr, t); attn->step(lr, t);
    norm2->step(lr, t); ffn->step(lr, t);
}


In [ ]:
%%writefile GPTModel.cuh
#pragma once
#include "Layer.cuh"
#include "EmbeddingLayer.cuh"
#include "TransformerBlock.cuh"
#include "RMSNormLayer.cuh"
#include "LinearLayer.cuh"
#include <vector>
class GPTModel : public Layer {
  EmbeddingLayer* emb; std::vector<TransformerBlock*> blocks;
  RMSNormLayer* fnorm; LinearLayer* lm_head;
  int vs, ed, bs, cs, nb;
public:
  GPTModel(int vs, int ed, int bs, int cs, int nb);
  ~GPTModel();
  float* forward(cublasHandle_t h, void* inp, int act) override;
  float* backward(cublasHandle_t h, float* dY) override;
  void step(float lr, int t) override;
};


In [ ]:
%%writefile GPTModel.cu
#include "GPTModel.cuh"

GPTModel::GPTModel(int vs, int ed, int bs, int cs, int nb) : Layer(bs, cs, ed) {
    this->vs = vs; this->ed = ed; this->bs = bs; this->cs = cs; this->nb = nb;
    emb = new EmbeddingLayer(vs, ed, bs, cs);
    for (int i = 0; i < nb; i++) blocks.push_back(new TransformerBlock(bs, cs, ed));
    fnorm = new RMSNormLayer(bs, cs, ed);
    lm_head = new LinearLayer(bs * cs, ed, vs);
}
GPTModel::~GPTModel() {
    delete emb; for (auto* b : blocks) delete b; delete fnorm; delete lm_head;
}

float* GPTModel::forward(cublasHandle_t h, void* inp, int act) {
    float* out = emb->forward(h, inp, ACTIVATION_NONE);
    for (auto* b : blocks) out = b->forward(h, out, ACTIVATION_NONE);
    out = fnorm->forward(h, out, ACTIVATION_NONE);
    return lm_head->forward(h, out, ACTIVATION_NONE);
}

float* GPTModel::backward(cublasHandle_t h, float* dY) {
    float* grad = lm_head->backward(h, dY);
    grad = fnorm->backward(h, grad);
    for (int i = nb - 1; i >= 0; i--) grad = blocks[i]->backward(h, grad);
    emb->backward(h, grad);
    return nullptr;
}

void GPTModel::step(float lr, int t) {
    emb->step(lr, t);
    for (auto* b : blocks) b->step(lr, t);
    fnorm->step(lr, t);
    lm_head->step(lr, t);
}


In [ ]:
%%writefile DataLoader.h
#pragma once
#include <string>
#include <vector>
#include <map>
class DataLoader {
  std::string raw_text; std::vector<int> tokens;
  std::map<char,int> c2i; std::map<int,char> i2c;
  int bs, cs, vs;
public:
  DataLoader(const std::string& fp, int bs, int cs);
  ~DataLoader();
  void get_batch(int* X, int* Y);
  int get_vocab_size() const { return vs; }
  int get_num_tokens() const { return (int)tokens.size(); }
  std::map<int,char> get_i2c() const { return i2c; }
};


In [ ]:
%%writefile DataLoader.cpp
#include "DataLoader.h"
#include <fstream>
#include <sstream>
#include <iostream>
#include <set>
#include <cstdlib>

DataLoader::DataLoader(const std::string& fp, int bs, int cs) {
    this->bs = bs; this->cs = cs;
    std::ifstream file(fp);
    if (!file.is_open()) { std::cerr << "ERROR: cannot open " << fp << std::endl; exit(1); }
    std::stringstream buf; buf << file.rdbuf(); raw_text = buf.str(); file.close();
    std::set<char> uniq(raw_text.begin(), raw_text.end());
    vs = uniq.size();
    int i = 0;
    for (char c : uniq) { c2i[c] = i; i2c[i] = c; i++; }
    tokens.reserve(raw_text.size());
    for (char c : raw_text) tokens.push_back(c2i[c]);
    std::cout << "DataLoader: " << tokens.size() << " chars, vocab=" << vs << std::endl;
}
DataLoader::~DataLoader() {}
void DataLoader::get_batch(int* X, int* Y) {
    for (int b = 0; b < bs; b++) {
        int start = rand() % (tokens.size() - cs - 1);
        for (int i = 0; i < cs; i++) {
            X[b * cs + i] = tokens[start + i];
            Y[b * cs + i] = tokens[start + i + 1];
        }
    }
}


In [ ]:
%%writefile main.cu
#include <iostream>
#include <cublas_v2.h>
#include "GPTModel.cuh"
#include "DataLoader.h"
#include <vector>
#include <string>
#include <cmath>
#include <algorithm>
#include <fstream>
#include <ctime>

// ─── NaN detection ────────────────────────────────────────────────────
__global__ void check_nan_kernel(float* t, int size, const char* name) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < size) {
        if (isnan(t[idx]) || isinf(t[idx]))
            printf("ALERT: NaN/Inf in %s at idx %d\n", name, idx);
    }
}

// ─── Gradient clipping ────────────────────────────────────────────────
__global__ void clip_gradients_kernel(float* dY, float min_v, float max_v, int total) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < total) {
        float v = dY[idx];
        if (v > max_v) v = max_v;
        if (v < min_v) v = min_v;
        if (isnan(v)) v = 0.0f;
        dY[idx] = v;
    }
}

// ─── Fused Softmax + Cross-Entropy backward ───────────────────────────
__global__ void cross_entropy_backward_kernel(float* logits, int* targets, float* dY,
                                               int vocab_size, int total_words) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < total_words) {
        int target = targets[idx];
        float max_val = -1e9f;
        for (int i = 0; i < vocab_size; i++)
            max_val = fmaxf(max_val, logits[idx * vocab_size + i]);

        float sum_exp = 0.0f;
        for (int i = 0; i < vocab_size; i++)
            sum_exp += expf(logits[idx * vocab_size + i] - max_val);

        for (int i = 0; i < vocab_size; i++) {
            float prob = expf(logits[idx * vocab_size + i] - max_val) / (sum_exp + 1e-7f);
            dY[idx * vocab_size + i] = (prob - (i == target ? 1.0f : 0.0f)) / (float)total_words;
        }
    }
}

// ─── Text generation ──────────────────────────────────────────────────
void generate_text(cublasHandle_t handle, GPTModel* model, std::string prompt,
                   int length, char* i2c, int cs, int vs) {
    std::cout << "\nPrompt: \"" << prompt << "\"\nGenerated: " << prompt;

    std::vector<int> ctx;
    for (char c : prompt) {
        int tok = 0;
        for (int v = 0; v < vs; v++) { if (i2c[v] == c) { tok = v; break; } }
        ctx.push_back(tok);
    }

    int* d_X; cudaMalloc(&d_X, sizeof(int) * cs);
    float* h_logits = (float*)malloc(sizeof(float) * cs * vs);

    for (int i = 0; i < length; i++) {
        std::vector<int> win;
        int start = std::max(0, (int)ctx.size() - cs);
        for (int j = start; j < (int)ctx.size(); j++) win.push_back(ctx[j]);
        while ((int)win.size() < cs) win.push_back(0);

        cudaMemcpy(d_X, win.data(), sizeof(int) * cs, cudaMemcpyHostToDevice);
        float* d_logits = model->forward(handle, d_X, 0);
        cudaMemcpy(h_logits, d_logits, sizeof(float) * cs * vs, cudaMemcpyDeviceToHost);

        int off = (cs - 1) * vs;
        float max_l = -1e9f;
        for (int v = 0; v < vs; v++) max_l = std::max(max_l, h_logits[off + v]);

        float sum_exp = 0.0f;
        std::vector<float> probs(vs);
        for (int v = 0; v < vs; v++) {
            probs[v] = expf(h_logits[off + v] - max_l);
            sum_exp += probs[v];
        }

        float r = ((float)rand() / RAND_MAX) * sum_exp;
        float cum = 0.0f;
        int next = 0;
        for (int v = 0; v < vs; v++) { cum += probs[v]; if (r <= cum) { next = v; break; } }

        std::cout << i2c[next] << std::flush;
        ctx.push_back(next);
    }
    std::cout << std::endl;
    cudaFree(d_X); free(h_logits);
}

// ─── MAIN ─────────────────────────────────────────────────────────────
int main() {
    cublasHandle_t handle;
    cublasCreate(&handle);

    // ── Hyperparameters ───────────────────────────────────────────────
    int context_size = 32;
    int batch_size = 128;
    int embedding_dim = 128;
    int num_blocks = 4;
    float base_lr = 3e-4f;
    int total_iterations = 10000;
    int warmup_steps = 1000;
    int log_every = 50;
    int eval_every = 500;

    std::cout << "=== GPT-CUDA v2 (Fixed) ===\n";
    std::cout << "Context: " << context_size << ", Batch: " << batch_size
              << ", Emb: " << embedding_dim << ", Blocks: " << num_blocks << "\n";
    std::cout << "Base LR: " << base_lr << ", Warmup: " << warmup_steps
              << ", Iterations: " << total_iterations << "\n";

    // ── Data ──────────────────────────────────────────────────────────
    DataLoader dataloader("input.txt", batch_size, context_size);
    int vocab_size = dataloader.get_vocab_size();

    // ── Model ─────────────────────────────────────────────────────────
    GPTModel model(vocab_size, embedding_dim, batch_size, context_size, num_blocks);
    int total_words = batch_size * context_size;

    // ── GPU memory ────────────────────────────────────────────────────
    int *h_X = (int*)malloc(sizeof(int) * total_words);
    int *h_targets = (int*)malloc(sizeof(int) * total_words);
    int *d_X, *d_targets; float *d_dY;
    cudaMalloc(&d_X, sizeof(int) * total_words);
    cudaMalloc(&d_targets, sizeof(int) * total_words);
    cudaMalloc(&d_dY, sizeof(float) * total_words * vocab_size);

    // ── Logging ───────────────────────────────────────────────────────
    std::ofstream log_file("training_log.csv");
    log_file << "iteration,loss,lr\n";

    std::cout << "\n=== TRAINING ===\n";
    time_t start_time = time(0);

    for (int iter = 0; iter < total_iterations; iter++) {
        // FIX #4: Learning rate warmup (linear)
        float lr;
        if (iter < warmup_steps) {
            lr = base_lr * ((float)(iter + 1) / (float)warmup_steps);
        } else {
            // Cosine decay after warmup
            float progress = (float)(iter - warmup_steps) / (float)(total_iterations - warmup_steps);
            lr = base_lr * 0.5f * (1.0f + cosf(3.14159265f * progress));
        }

        dataloader.get_batch(h_X, h_targets);
        cudaMemcpy(d_X, h_X, sizeof(int) * total_words, cudaMemcpyHostToDevice);
        cudaMemcpy(d_targets, h_targets, sizeof(int) * total_words, cudaMemcpyHostToDevice);

        // Forward
        float* d_logits = model.forward(handle, d_X, 0);

        // Loss computation (every log_every steps)
        if (iter % log_every == 0) {
            float* h_logits = (float*)malloc(sizeof(float) * total_words * vocab_size);
            cudaMemcpy(h_logits, d_logits, sizeof(float) * total_words * vocab_size, cudaMemcpyDeviceToHost);

            float loss = 0.0f;
            for (int i = 0; i < total_words; i++) {
                int target = h_targets[i];
                float max_l = -1e9f;
                for (int v = 0; v < vocab_size; v++)
                    max_l = std::max(max_l, h_logits[i * vocab_size + v]);

                float sum_exp = 0.0f;
                for (int v = 0; v < vocab_size; v++)
                    sum_exp += expf(h_logits[i * vocab_size + v] - max_l);

                float prob = expf(h_logits[i * vocab_size + target] - max_l) / sum_exp;
                loss += -logf(prob + 1e-7f);
            }
            loss /= total_words;
            free(h_logits);

            time_t elapsed = time(0) - start_time;
            std::cout << "[" << iter << "/" << total_iterations << "] loss="
                      << loss << " lr=" << lr << " time=" << elapsed << "s\n";
            log_file << iter << "," << loss << "," << lr << "\n";
        }

        // Backward
        int threads = 256;
        int blocks_ce = (total_words + threads - 1) / threads;
        cross_entropy_backward_kernel<<<blocks_ce, threads>>>(
            d_logits, d_targets, d_dY, vocab_size, total_words);

        // Gradient clipping
        int total_grad = total_words * vocab_size;
        int blocks_clip = (total_grad + threads - 1) / threads;
        clip_gradients_kernel<<<blocks_clip, threads>>>(d_dY, -1.0f, 1.0f, total_grad);
        cudaDeviceSynchronize();

        model.backward(handle, d_dY);
        model.step(lr, iter + 1);  // iter+1 for Adam bias correction

        // NaN check (every 500 steps)
        if (iter % 500 == 0) {
            int total_logits = total_words * vocab_size;
            check_nan_kernel<<<(total_logits + 255)/256, 256>>>(d_logits, total_logits, "logits");
            cudaDeviceSynchronize();
        }
    }

    log_file.close();
    std::cout << "\n=== TRAINING COMPLETE ===\n";

    // ── Generation ────────────────────────────────────────────────────
    std::map<int, char> i2c_map = dataloader.get_i2c();
    char* i2c = (char*)malloc(sizeof(char) * vocab_size);
    for (int i = 0; i < vocab_size; i++) i2c[i] = i2c_map[i];

    std::cout << "\n=== TEXT GENERATION ===\n";
    generate_text(handle, &model, "First Citizen:", 100, i2c, context_size, vocab_size);
    generate_text(handle, &model, "ROMEO:", 100, i2c, context_size, vocab_size);
    generate_text(handle, &model, "The king", 100, i2c, context_size, vocab_size);

    // Cleanup
    free(h_X); free(h_targets); free(i2c);
    cudaFree(d_X); cudaFree(d_targets); cudaFree(d_dY);
    cublasDestroy(handle);

    return 0;
}


In [ ]:
!nvcc -O3 -o gpt_cuda main.cu GPTModel.cu TransformerBlock.cu AttentionLayer.cu FeedForwardLayer.cu RMSNormLayer.cu EmbeddingLayer.cu LinearLayer.cu DataLoader.cpp -lcublas -Wno-deprecated-gpu-targets 2>&1
import os
assert os.path.exists('gpt_cuda'), 'Compilation failed!'
print('✓ Compilation successful — gpt_cuda binary ready')

In [ ]:
!./gpt_cuda

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('training_log.csv')
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

ax1.plot(df['iteration'], df['loss'])
ax1.set_xlabel('Iteration'); ax1.set_ylabel('Cross-Entropy Loss')
ax1.set_title('Training Loss'); ax1.grid(True, alpha=0.3)

ax2.plot(df['iteration'], df['lr'])
ax2.set_xlabel('Iteration'); ax2.set_ylabel('Learning Rate')
ax2.set_title('LR Schedule (warmup + cosine decay)'); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_curve.png', dpi=100, bbox_inches='tight')
plt.show()
print('✓ Plot saved as training_curve.png')